# Kunci Lab 0: Mulai dari Nol

Jawaban latihan Lab 0.



## Persiapan: alamat data



## Cara mengikuti notebook ini

Jalankan sel berurutan dengan `Shift + Enter`. Setiap notebook dapat
dimulai sendiri dengan koneksi internet yang aktif. Sel persiapan cukup dijalankan
apa adanya; fokus latihan dimulai saat tabel data muncul.

Baca penjelasan sebelum menjalankan kode, lalu sebutkan arti hasilnya.
Ubah satu hal setiap mencoba: nama kolom pada X, pilihan model, atau nilai prediksi.
Baris `#` adalah catatan. `=` menyimpan hasil dan `==` membandingkan nilai.
Contoh `tabel["kolom"]` berarti memilih satu kolom dari tabel.

Jika muncul error merah, berhenti di sel itu dan baca baris terakhirnya.
Jika ada warning, baca pesannya dan periksa hasil estimasi sebelum lanjut.
Panduan simbol dan contoh tambahan ada di `panduan/01_panduan_pemula.md`.



### Pola pemodelan yang dipakai

`X` adalah tabel fitur dan `y` adalah kolom target. Model utama memakai
scikit-learn: buat model, jalankan `fit(X, y)`, lalu gunakan `predict(X_baru)`.
Koefisien ada pada `coef_` dan intersep pada `intercept_`. Tabel `koefisien_...`
hanya memberi nama kolom agar angka mudah dibaca. `score(X, y)` pada regresi
linear adalah R² pada data yang diberikan; hasil pada data latih belum mengukur
kemampuan memprediksi data baru.

Nama `uji_...` menandai pelengkap statistik untuk p-value, standard error, atau
uji asumsi. Perhitungannya memakai X dan y yang sama. Bagian model khusus diberi
catatan tersendiri ketika scikit-learn belum menyediakan estimatornya.



In [1]:
URL_DATA = "https://raw.githubusercontent.com/akhiyarwaladi/ekonometrika_coding_workshop/main/data/"

print("Alamat data siap. Jalankan sel pembacaan CSV berikutnya.")



Alamat data siap. Jalankan sel pembacaan CSV berikutnya.


In [2]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import statsmodels.api as sm



## Latihan 1 dan 2: baca data dan hitung rata-rata



<!-- kamus_variabel:uts_4.csv -->

### Kenali data: Angka harapan hidup

**Data:** `data/uts_4.csv`. **Satu baris:** satu negara pada contoh latihan.

| Variabel/kolom | Pengertian | Satuan atau kode | Peran |
|---|---|---|---|
| `Y_Angka_Harapan_Hidup` | Rata-rata lama hidup yang diharapkan menurut indikator pada contoh. | Tahun | Y: target |
| `X1_Pengeluaran_Kesehatan` | Pengeluaran kesehatan per penduduk (per kapita). | Uang per kapita; mata uang/periode tidak dirinci | X: penjelas |
| `X2_Tingkat_Imunisasi` | Persentase anak yang memperoleh imunisasi. | Persen; jenis imunisasi tidak dirinci | X: penjelas |
| `X3_Konsumsi_Kalori` | Rata-rata konsumsi energi makanan per hari. | Kalori harian; rincian satuan energi mengikuti sumber | X: penjelas |

Sumber definisi: [rujukan data](https://paggalih.github.io/matkul/sem2025-01/ekonometrika/).



In [3]:
harapan_hidup = pd.read_csv(URL_DATA + "uts_4.csv")
print("Jumlah observasi:", len(harapan_hidup))
harapan_hidup.head()



Jumlah observasi: 50


,Y_Angka_Harapan_Hidup,X1_Pengeluaran_Kesehatan,X2_Tingkat_Imunisasi,X3_Konsumsi_Kalori
0,75.5,1550,92,2850
1,72.1,850,85,2600
2,81.2,4500,98,3300
3,82.5,5500,99,3450
4,68.5,350,75,2300


In [4]:
print(
    "Rata-rata harapan hidup :", round(harapan_hidup["Y_Angka_Harapan_Hidup"].mean(), 2)
)
print(
    "Rata-rata imunisasi     :", round(harapan_hidup["X2_Tingkat_Imunisasi"].mean(), 2)
)



Rata-rata harapan hidup : 76.95
Rata-rata imunisasi     : 91.86


## Latihan 3: regresi sederhana



In [5]:
sampel_model = harapan_hidup[
    ["Y_Angka_Harapan_Hidup", "X1_Pengeluaran_Kesehatan"]
].dropna()
X_model = sampel_model[["X1_Pengeluaran_Kesehatan"]]
y_model = sampel_model["Y_Angka_Harapan_Hidup"]

model = LinearRegression()
model.fit(X_model, y_model)

koefisien_model = pd.Series(model.coef_, index=X_model.columns)
koefisien_model.loc["Intercept"] = model.intercept_

# Pelengkap statistik: standard error dan pengujian pada sampel yang sama.
uji_model = sm.OLS(
    y_model,
    sm.add_constant(X_model, has_constant="add").rename(columns={"const": "Intercept"}),
).fit()
print(koefisien_model.round(5))
print(uji_model.pvalues.round(4))



X1_Pengeluaran_Kesehatan     0.00243
Intercept                   70.55237
dtype: float64
Intercept                   0.0
X1_Pengeluaran_Kesehatan    0.0
dtype: float64


Kesimpulan: koefisien pengeluaran kesehatan 0,00243 dengan p-value
0,0000, jadi signifikan. Setiap tambahan satu unit pengeluaran kesehatan
diiringi kenaikan harapan hidup 0,00243 tahun. R kuadrat model ini
sekitar 0,91.
